# Day 5: System Cursor

## Focus Area
Multi file edit agents, diff application, and the agent mode loop.

## Core Problem Being Solved
A single file autocomplete tool can only reason about one file at a time. Real world refactors, like renaming a function or updating a schema, affect multiple files at once. If a model edits without awareness of related files, it silently leaves other files outdated. Cursor's core design challenge is coordinating change across a codebase without a human manually tracing every dependency.

## The Plan, Diff, Apply, Verify Loop

### 1. Plan Stage
The model first decides which files need to change and why, before generating any actual code. This decision is informed by retrieval: a codebase index built from embeddings and a symbol graph, which shows where a given function or variable is referenced across the project.

### 2. Diff Stage
The model generates the actual change as a diff, not a full file rewrite. A diff is a machine readable instruction that tells a program exactly what to remove and what to add. This is necessary because a program cannot understand vague natural language instructions the way a human can. It needs an exact, unambiguous instruction.

### 3. Apply Stage
The generated diff is applied to the actual file using a deterministic matching process. The engine searches for the exact original content in the file and replaces it with the new content.

### 4. Verify Stage
After applying, the system checks whether the result is valid, for example by compiling the code, checking for syntax errors, or running existing tests. If verification fails, the system loops back with error feedback so the model can retry.

## Understanding Diff Format

A diff is not just a visual red and green display. That display is only a representation shown to humans. The actual diff is a structured text instruction meant for a program to read and execute mechanically.

### SEARCH and REPLACE Block
This is the common format used to represent a diff:

The SEARCH block contains the exact original content as it currently exists in the file. The engine uses this block to locate the precise location that needs to change.

The REPLACE block contains the new content that should take the place of the SEARCH block once found.

This works similarly to a find and replace feature in a document editor. The find box is like the SEARCH block, and the replace box is like the REPLACE block. The difference is that here entire code sections are matched instead of single words.

### Why the Full Function Is Given Instead of Just One Line
A single line inside the SEARCH block may not be unique across a file. Giving more surrounding lines makes the match unique, so the engine does not get confused about which occurrence to replace.

## Why Whitespace and Exact Matching Matter

The underlying matching mechanism is a literal string comparison, not an intelligent one. If a model reproduces a snippet with a small difference, like a tab instead of spaces or an extra blank line, the exact match will fail even though the content looks the same to a human eye.

## Fuzzy Matching

Because models sometimes reproduce a snippet with tiny inconsistencies, systems use fuzzy matching to still recognize it as the same content. This generally works through:

1. Normalizing whitespace before comparing, so spacing differences do not cause failure.
2. Calculating a similarity score between the SEARCH block and the actual file content. If the similarity crosses a set threshold, it is accepted as a match.
3. If similarity is too low, or if the same content matches in more than one place in the file, the match is considered ambiguous.

## Handling Ambiguous or Failed Matches

If a SEARCH block matches in more than one location in a file, or does not match confidently anywhere, the correct behavior is to treat it as ambiguous and send the situation back to the model with an error, rather than guessing or forcefully applying a change. This prevents wrong or unintended edits from silently happening.

## Multi File Consistency Checking

When multiple files are edited together, all the diffs must remain consistent with one another. For example, if a function's signature changes in one file, every other file calling that function must also be updated the same way.

To manage this, systems typically:
- Build a dependency graph at the symbol level, showing which functions call which, and which files import which symbols.
- Include all affected files together in the planning stage so the model sees the complete picture at once instead of working file by file blindly.
- Run a cross file static check after applying changes, such as a type checker or an import resolver, to catch any broken references.

## Why the Planning Stage Graph Is Not Enough on Its Own

The dependency graph used during planning is a prediction, not a guarantee. It can be incomplete because of:
- Dynamic behavior, where a function is called through a variable or string rather than a direct reference, which static analysis cannot always detect.
- Reflection or configuration driven wiring where the connection between files is not explicitly written in code.
- A stale index, where the codebase map has not been updated with the latest changes.

Because of this, a file that was never touched or included in the plan can still silently break if it depended on something that changed elsewhere.

## Why the Verify Stage Is the Real Safety Net

Since the planning stage graph can miss things, the verify stage acts as the actual safety net that checks reality rather than predictions. This includes:
- A compiler or type checker that will catch a broken reference even if that file was not part of the original plan.
- Running the test suite to catch logic breakage.
- Human review of the diff before it is finalized, as a final semantic check that automated tools might miss.

A simple way to remember this: the planning stage trusts the map, while the verification stage trusts the territory. The map is never as complete as the real system.

## Human in the Loop as a Safety Layer

Cursor intentionally shows the diff to a human for review before applying it, instead of silently auto applying changes. This exists because automated verification, like passing tests or successful compilation, does not guarantee that the change is logically or semantically correct. A model can be confidently wrong in a way that still technically compiles. Human review catches this last category of mistake.

However, human review cannot mean manually checking everything, because that would remove the entire benefit of automation. In practice, only a small percentage of outputs should require human review, while most should pass through automated triage checks.

## Automated Triage Ideas Discussed for the RFQ Agent

Instead of relying only on manual review for every output, layered automated checks can reduce load on humans while keeping safety intact:

1. Confidence or agreement check: If two different specialist models evaluate the same input and their outputs agree, it can be trusted automatically. If they disagree, it gets flagged for review.
2. Field level consistency check: Programmatically compare extracted values, like quantity or price, against the original source document to confirm they match structurally.
3. Anomaly or outlier detection: If an evaluation is far outside a normal historical range, it gets automatically flagged instead of passed through directly.

## Who Actually Generates the Diff

The model itself generates both the decision of which files to change and the actual diff content. This happens because during the planning stage, retrieval brings the relevant file content into the model's context, and the model reasons over that content to decide what needs to change and how. The diff format is simply the structured way the model is instructed to express that change so it can be applied mechanically afterward.

## Why the Model Cannot Be Fully Trusted to Catch Everything on Its Own

Even if a model appears capable of noticing duplicate or similar code while assembling context, this cannot be relied on alone because:

1. The model may not have the full codebase in context due to context window limits. Retrieval only brings in relevant chunks, not the entire project, so a duplicate elsewhere may never even be seen.
2. The model is a probabilistic reasoner, not a guaranteed deterministic checker. It might miss subtle duplicates even when they are technically visible to it.
3. Relying on a single layer of protection is risky. Having both model level awareness and a deterministic apply stage check creates layered protection, so a mistake at one layer does not silently cause damage.

## General Production Engineering Principles To Keep In Mind

1. Idempotency: Any retry logic must be safe to repeat without causing duplicate side effects, such as duplicate database entries.

2. Build observability from the start: Log reasoning traces for every agent decision, including which model was chosen and why, so failures can be understood later without needing to reproduce them.

3. Categorize failure types:
   - Recoverable failures that retries can fix.
   - Non recoverable but detectable failures that validation layers can catch.
   - Non recoverable and silent failures, which are the most dangerous and should be minimized through design.

4. Confidence is not the same as correctness: A model stating it is confident does not guarantee the output is correct. Deterministic structural checks should exist wherever possible, independent of model confidence.

5. Design for graceful degradation: If one part of the pipeline fails, such as one specialist model being unavailable, the system should fall back to an alternative rather than crashing entirely.

6. Treat cost and latency as real design constraints: Routing and triage logic exist not only for accuracy but also to avoid unnecessary expensive model calls at scale.

7. Silent failure is worse than loud failure: A system crashing immediately reveals a problem. A system confidently producing a wrong result without any flag can cause damage long before anyone notices.

## Cross Question Answered

**What stops a multi file agentic edit from silently breaking a file it did not even touch?**

The planning stage dependency graph narrows down likely affected files, but it can be incomplete due to dynamic behavior or a stale index. The real protection comes from the verification stage, where compilers, type checkers, and test suites check the actual current state of the codebase rather than a predicted map. Additionally, treating any ambiguous match during diff application as an error instead of a silent guess adds another layer of protection. Together, these layers form a defense in depth approach, since no single layer alone is completely reliable.